In [ ]:
import cv2
import time
from ultralytics import YOLO
from pathlib import Path
import os

# --- CONFIGURATION ---
# 1. Path to best trained YOLOv8 model
MODEL_PATH = Path("../runs/detect/ppe_final_tuned_model2/weights/best.pt")

# 2. Video source: 0 for webcam, or path to a video file
VIDEO_SOURCE = Path("../dataset/source_files/hardhat.mp4")

# 3. Directory to save the output video
OUTPUT_DIR = Path("../outputs")

# 4. Confidence threshold for detections
CONFIDENCE_THRESHOLD = 0.5

# 5. Classes that constitute a violation
VIOLATION_CLASSES = {'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest'}

# 6. Alerting configuration
ALERT_COOLDOWN_SECONDS = 5  # Wait 5 seconds before sending another alert


def main():
    """
    Main function to run the real-time PPE detection, display it, and save the output.
    """
    # --- SETUP ---
    # Create the output directory if it doesn't exist
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Load the trained YOLOv8 model
    if not MODEL_PATH.exists():
        print(f"Error: Model file not found at '{MODEL_PATH}'")
        return
    model = YOLO(MODEL_PATH)

    # Initialize video capture
    try:
        # Convert Path object to string for VideoCapture
        video_source_str = str(VIDEO_SOURCE) if VIDEO_SOURCE != 0 else 0
        cap = cv2.VideoCapture(video_source_str)
        if not cap.isOpened():
            raise IOError(f"Cannot open video source: {VIDEO_SOURCE}")
    except Exception as e:
        print(f"Error initializing video capture: {e}")
        return

    # --- VIDEO WRITER SETUP ---
    # Get video properties for the writer
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Define the output video path
    if video_source_str == 0:
        # For webcam, create a timestamped filename
        output_filename = f"webcam_output_{time.strftime('%Y%m%d_%H%M%S')}.mp4"
    else:
        # For video file, use its name with a suffix
        output_filename = f"{VIDEO_SOURCE.stem}_result.mp4"

    output_path = OUTPUT_DIR / output_filename

    # Initialize the VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4 file
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (frame_width, frame_height))

    last_alert_time = 0

    print("Starting real-time detection... Press 'q' to quit.")
    print(f"Output video will be saved to: {output_path}")

    # --- MAIN PROCESSING LOOP ---
    while True:
        success, frame = cap.read()
        if not success:
            if VIDEO_SOURCE != 0:
                print("End of video file reached.")
            else:
                print("Failed to capture frame from webcam.")
            break

        # Run YOLOv8 inference on the frame
        results = model(frame, stream=True, verbose=False)

        violations_in_frame = set()

        for r in results:
            boxes = r.boxes
            for box in boxes:
                # --- Get detection data ---
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                confidence = box.conf[0]
                class_id = int(box.cls[0])
                class_name = model.names[class_id]

                # --- Filter weak detections ---
                if confidence < CONFIDENCE_THRESHOLD:
                    continue

                # --- Rule Engine: Check for violations ---
                is_violation = class_name in VIOLATION_CLASSES
                if is_violation:
                    violations_in_frame.add(class_name)

                # --- Detection Overlay ---
                color = (0, 0, 255) if is_violation else (0, 255, 0)  # Red for violation, Green for safe
                label = f"{class_name}: {confidence:.2f}"
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # --- Alerting Mechanism ---
        current_time = time.time()
        if violations_in_frame and (current_time - last_alert_time > ALERT_COOLDOWN_SECONDS):
            violation_str = ", ".join(sorted(list(violations_in_frame)))
            print(f"ALERT: Safety violation(s) detected: [{violation_str}] at {time.ctime()}!")
            last_alert_time = current_time
            # Visual alert on the screen
            cv2.putText(frame, "!!! VIOLATION DETECTED !!!", (50, 50),
                        cv2.FONT_HERSHEY_TRIPLEX, 1.0, (0, 0, 255), 2)

        # Display the output frame
        cv2.imshow("Real-Time PPE Detection", frame)

        # Write the processed frame to the output video file
        writer.write(frame)

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    # --- CLEANUP ---
    cap.release()
    writer.release()  # Release the video writer
    cv2.destroyAllWindows()
    print(f"Detection stopped. Video successfully saved to '{output_path}'.")

if __name__ == "__main__":
    main()

Starting real-time detection... Press 'q' to quit.
Output video will be saved to: ..\outputs\hardhat_result.mp4
ALERT: Safety violation(s) detected: [NO-Hardhat] at Mon Sep 29 21:45:25 2025!
ALERT: Safety violation(s) detected: [NO-Mask, NO-Safety Vest] at Mon Sep 29 21:45:30 2025!
ALERT: Safety violation(s) detected: [NO-Safety Vest] at Mon Sep 29 21:45:35 2025!
ALERT: Safety violation(s) detected: [NO-Safety Vest] at Mon Sep 29 21:45:41 2025!
ALERT: Safety violation(s) detected: [NO-Safety Vest] at Mon Sep 29 21:45:46 2025!
ALERT: Safety violation(s) detected: [NO-Mask] at Mon Sep 29 21:45:51 2025!
ALERT: Safety violation(s) detected: [NO-Mask, NO-Safety Vest] at Mon Sep 29 21:45:57 2025!
ALERT: Safety violation(s) detected: [NO-Mask] at Mon Sep 29 21:46:02 2025!
ALERT: Safety violation(s) detected: [NO-Mask, NO-Safety Vest] at Mon Sep 29 21:46:07 2025!
End of video file reached.
Detection stopped. Video successfully saved to '..\outputs\hardhat_result.mp4'.
